# Проверка гипотезы и прогноз нагрузки по обращениям

Этот notebook помогает пройти полный аналитический путь: от проверки качества исходных данных до статистического теста, временного ряда, простого baseline-прогноза, ARIMA-прогноза и итогового вывода.

## Как работать с notebook

1. Выполняйте ячейки строго сверху вниз.
2. Перед запуском следующего блока прочитайте пояснение: **что подаём на вход, что делаем, что получаем и как проверяем результат**.
3. После каждой контрольной точки сравнивайте фактический результат с описанием.
4. Не меняйте исходный CSV. Все преобразования выполняются в копии таблицы.
5. Если появляется ошибка, сначала прочитайте её последнюю строку: обычно там указана основная причина.

В результате работы вы:

- сравните время решения обращений в каналах `email` и `chat`;
- проверите гипотезу с помощью t-test;
- построите недельный временной ряд;
- рассчитаете скользящее среднее;
- построите baseline-прогноз;
- оцените прогноз с помощью MAE и RMSE;
- сформулируете аналитический вывод с ограничениями.


## 1. Рабочая ситуация

Вы работаете аналитиком в команде поддержки клиентов. Руководителю нужны ответы на два вопроса:

1. Отличается ли среднее время решения обращений между каналами `email` и `chat`?
2. Какой может быть ожидаемая недельная нагрузка на поддержку, если ориентироваться на прошлую динамику?

Эти вопросы требуют разных подходов:

- для сравнения двух групп используется статистический тест;
- для анализа динамики используется временной ряд;
- для оценки будущей нагрузки сначала строится простой baseline-прогноз.

Важно: итогом работы является не только число или график, но и корректная интерпретация результата.


## 2. Проверка окружения

### Что делает этот шаг

Проверяет:

- версию Python;
- текущую рабочую папку;
- наличие необходимых библиотек;
- версии основных библиотек, включая `statsmodels` для ARIMA.

### Почему это важно

Если библиотека не установлена или notebook открыт не из той папки, дальнейшие ячейки могут завершиться ошибкой. Этот блок помогает найти техническую причину до начала анализа.

### Что должно получиться

В выводе должны появиться:

- строка с версией Python;
- путь к текущей папке;
- версии `pandas` и `numpy`;
- отсутствие сообщения `Не удалось импортировать библиотеку`.


In [ ]:
# Path помогает безопасно работать с путями к папкам и файлам на разных ОС.
from pathlib import Path

# sys нужен, чтобы вывести версию Python и проверить окружение.
import sys

# Показываем версию Python: она полезна при диагностике несовместимости библиотек.
print("Python:", sys.version)

# Path.cwd() возвращает папку, из которой сейчас запущен notebook.
print("Рабочая папка:", Path.cwd())

try:
    # pandas используется для загрузки, очистки и агрегации табличных данных.
    import pandas as pd

    # NumPy нужен для математических операций, в том числе для квадратного корня RMSE.
    import numpy as np

    # matplotlib используется для построения графиков.
    import matplotlib.pyplot as plt

    # scipy.stats содержит статистические тесты.
    from scipy import stats

    # Метрики scikit-learn используются для оценки ошибки прогноза.
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    # statsmodels содержит модель ARIMA для временных рядов.
    import statsmodels
    from statsmodels.tsa.arima.model import ARIMA

    # Печатаем версии основных библиотек, чтобы результат можно было воспроизвести.
    print("pandas:", pd.__version__)
    print("numpy:", np.__version__)
    print("statsmodels:", statsmodels.__version__)
except ImportError as error:
    # Этот блок сработает, если хотя бы одна библиотека не установлена.
    print("Не удалось импортировать библиотеку:", error)
    print("Установите зависимости из requirements.txt или используйте Google Colab.")


## 3. Проверка файлов

### Что делает этот шаг

Ищет файл `support_tickets.csv` по относительному пути:

```text
data/raw/support_tickets.csv
```

Notebook поддерживает два варианта запуска:

- из корня проекта;
- из папки `notebooks`.

### Почему используются относительные пути

Относительный путь не привязан к конкретному пользователю или компьютеру. Поэтому notebook можно перенести на другой компьютер без изменения пути вида `C:\Users\...`.

### Что должно получиться

В выводе должна появиться строка `OK:` и полный путь к найденному CSV-файлу.

Если выводится `НЕ НАЙДЕН`, проверьте структуру проекта и убедитесь, что архив распакован полностью.


In [ ]:
# Определяем папку, из которой запущен notebook.
current = Path.cwd()

# Описываем относительный путь к исходному CSV внутри проекта.
relative_data_path = Path("data") / "raw" / "support_tickets.csv"

# Notebook можно запускать из корня проекта или из папки notebooks.
# В первом случае data/raw находится внутри current.
if (current / relative_data_path).exists():
    PROJECT_ROOT = current

# Во втором случае notebook открыт из папки notebooks,
# поэтому корень проекта находится на один уровень выше.
elif (current.parent / relative_data_path).exists():
    PROJECT_ROOT = current.parent

# Если файл не найден ни в одном месте, сохраняем current,
# чтобы ниже вывести понятную диагностику.
else:
    PROJECT_ROOT = current

# Формируем полный путь к исходному CSV.
DATA_PATH = PROJECT_ROOT / relative_data_path

# Формируем путь к папке для результатов.
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Создаём папку outputs, если её ещё нет.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("Проверяем файл:")

# Проверяем фактическое наличие файла до чтения.
if DATA_PATH.exists():
    print("OK:", DATA_PATH)
else:
    print("НЕ НАЙДЕН:", DATA_PATH)
    print("Проверьте, что файл support_tickets.csv лежит в data/raw внутри папки проекта.")


## 4. Загрузка данных

### Вход

CSV-файл с обращениями клиентов.

### Действие

`pd.read_csv()` читает файл и создаёт объект `DataFrame` — таблицу pandas.

### Выход

- размер таблицы в формате `(число строк, число столбцов)`;
- первые пять строк таблицы.

### Как проверить результат

Убедитесь, что:

- таблица не пустая;
- названия столбцов читаются корректно;
- в первых строках есть поля `ticket_id`, `created_at`, `channel`, `resolution_hours`;
- все данные не попали в один столбец.

После загрузки ещё нельзя делать содержательные выводы: сначала нужно проверить качество данных.


In [ ]:
# Читаем CSV-файл и создаём DataFrame tickets.
# На этом этапе значения загружаются без очистки, чтобы увидеть исходное состояние данных.
tickets = pd.read_csv(DATA_PATH)

# shape возвращает кортеж: (число строк, число столбцов).
print("Размер таблицы:", tickets.shape)

# head() показывает первые пять строк и помогает быстро проверить структуру таблицы.
display(tickets.head())


## 5. Первичная проверка качества

### Что проверяем

1. Типы данных в каждом столбце.
2. Количество пропусков.
3. Дубликаты идентификатора обращения.
4. Варианты написания каналов.

### Зачем это нужно

Статистический тест и временной ряд могут дать некорректный результат, если:

- числа прочитаны как текст;
- дата осталась строкой;
- одно обращение продублировано;
- значения `Email`, `email` и ` email ` считаются разными категориями;
- есть пропуски в анализируемом показателе.

### Что должно получиться

В выводе будут четыре диагностических блока. На этом шаге ничего не удаляется: мы только изучаем исходное состояние данных.


In [ ]:
# dtypes показывает, какой тип pandas назначил каждому столбцу.
print("Типы данных:")
display(tickets.dtypes)

# isna() отмечает пропуски, а sum() считает их по каждому столбцу.
print("Пропуски по столбцам:")
display(tickets.isna().sum())

# duplicated() ищет повторные ticket_id.
# keep='first' используется по умолчанию: первое вхождение не считается дубликатом.
print("Дубликаты ticket_id:", tickets["ticket_id"].duplicated().sum())

# value_counts() показывает, сколько строк относится к каждому варианту channel.
# dropna=False позволяет также увидеть пропуски, если они есть.
print("Каналы обращений:")
display(tickets["channel"].value_counts(dropna=False))


## 6. Подготовка данных

На этом этапе создаётся отдельная копия таблицы `tickets_clean`. Исходный DataFrame `tickets` сохраняется без изменений — это позволяет сравнить исходные и очищенные данные.

### Что делаем

1. Преобразуем `created_at` в тип даты и времени.
2. Некорректные даты превращаем в `NaT` — специальное обозначение отсутствующей даты.
3. Удаляем пробелы в начале и конце текстовых значений.
4. Приводим названия каналов и категорий к нижнему регистру.
5. Считаем количество проблемных значений до фильтрации.

### Контрольная логика

- дата должна быть распознана;
- `resolution_hours` должно быть больше нуля;
- один `ticket_id` должен соответствовать одному обращению;
- одинаковые категории должны иметь одинаковое написание.

Пока мы только нормализуем поля и считаем ошибки. Удаление проблемных строк выполняется в следующем блоке.


In [ ]:
# Создаём копию, чтобы исходный DataFrame tickets остался без изменений.
tickets_clean = tickets.copy()

# Преобразуем текстовые даты в datetime.
# errors='coerce' заменяет нераспознаваемые даты на NaT вместо остановки программы.
tickets_clean["created_at"] = pd.to_datetime(
    tickets_clean["created_at"],
    errors="coerce"
)

# Нормализуем текстовые поля:
# 1) приводим значения к строковому типу;
# 2) удаляем пробелы в начале и конце.
for col in ["channel", "priority", "category", "customer_segment", "region"]:
    tickets_clean[col] = tickets_clean[col].astype(str).str.strip()

# Приводим каналы к нижнему регистру: Email, EMAIL и email станут email.
tickets_clean["channel"] = tickets_clean["channel"].str.lower()

# Аналогично нормализуем категории обращений.
tickets_clean["category"] = tickets_clean["category"].str.lower()

# Считаем проблемы до удаления строк.
print("Некорректные даты:", tickets_clean["created_at"].isna().sum())
print("Некорректные resolution_hours <= 0:", (tickets_clean["resolution_hours"] <= 0).sum())
print("Дубликаты ticket_id:", tickets_clean["ticket_id"].duplicated().sum())


In [ ]:
# Удаляем повторные обращения по ticket_id.
# Первое вхождение сохраняется, последующие дубликаты удаляются.
tickets_clean = tickets_clean.drop_duplicates(subset=["ticket_id"]).copy()

# Формируем таблицу analysis_data только из строк, пригодных для анализа:
# - дата распознана;
# - время решения не пропущено;
# - время решения положительное.
analysis_data = tickets_clean[
    tickets_clean["created_at"].notna()
    & tickets_clean["resolution_hours"].notna()
    & (tickets_clean["resolution_hours"] > 0)
].copy()

# Сравниваем объём исходных и подготовленных данных.
print("Строк в исходной таблице:", len(tickets))
print("Строк после подготовки для анализа:", len(analysis_data))

# Проверяем, какие каналы и в каком количестве остались после очистки.
print("Каналы после очистки:")
display(analysis_data["channel"].value_counts())


## 7. Формулировка гипотезы

Гипотезы нужно записать **до просмотра результата теста**, чтобы не подстраивать формулировку под полученное p-value.

**H0 — нулевая гипотеза:** среднее время решения обращений в каналах `email` и `chat` не отличается.

**H1 — альтернативная гипотеза:** среднее время решения обращений в каналах `email` и `chat` отличается.

Это двусторонняя проверка: заранее не утверждается, какой именно канал быстрее. Проверяется сам факт различия средних значений.

Контрольный вопрос: какой столбец является числовым показателем, а какой — признаком группы?

- показатель: `resolution_hours`;
- группа: `channel`.


## 8. Описательное сравнение групп

До статистического теста нужно понять, как выглядят две группы.

### Что рассчитываем

- `tickets_count` — число уникальных обращений;
- `mean_resolution` — среднее время решения;
- `median_resolution` — медианное время решения;
- `std_resolution` — стандартное отклонение;
- `min_resolution` и `max_resolution` — диапазон значений.

### Почему одного среднего недостаточно

Две группы могут иметь разные средние, но большой разброс. Медиана помогает понять, не искажено ли среднее отдельными большими значениями. Размер групп показывает, достаточно ли наблюдений для сравнения.

### Что должно получиться

Таблица из двух строк: одна для `email`, другая для `chat`.


In [ ]:
# Указываем две группы, которые будем сравнивать.
compare_channels = ["email", "chat"]

# Оставляем только обращения из выбранных каналов.
group_data = analysis_data[
    analysis_data["channel"].isin(compare_channels)
].copy()

# Группируем строки по каналу и рассчитываем описательные показатели.
channel_summary = (
    group_data
    .groupby("channel", as_index=False)
    .agg(
        # Число уникальных обращений в группе.
        tickets_count=("ticket_id", "nunique"),

        # Среднее время решения.
        mean_resolution=("resolution_hours", "mean"),

        # Медиана — центральное значение, устойчивое к отдельным выбросам.
        median_resolution=("resolution_hours", "median"),

        # Стандартное отклонение показывает разброс значений вокруг среднего.
        std_resolution=("resolution_hours", "std"),

        # Минимум и максимум помогают увидеть диапазон.
        min_resolution=("resolution_hours", "min"),
        max_resolution=("resolution_hours", "max")
    )
)

# Ожидается таблица из двух строк: email и chat.
display(channel_summary)


## 9. Статистический тест

Для сравнения средних двух независимых групп используется `scipy.stats.ttest_ind`.

### Почему `equal_var=False`

Этот параметр включает вариант Welch t-test. Он не требует предполагать, что дисперсии в группах одинаковы, и поэтому является более безопасным базовым выбором для учебного сравнения независимых групп.

### Что возвращает тест

- `statistic` — t-статистику;
- `pvalue` — p-value.

### Правило решения

- если `p-value < 0.05`, есть основание отвергнуть H0;
- если `p-value >= 0.05`, недостаточно оснований отвергнуть H0.

Статистическая значимость не показывает практическую важность различия. После теста всё равно нужно смотреть на величину разницы и ограничения данных.


In [ ]:
# Выделяем время решения обращений из канала email.
# dropna() исключает пропуски из статистического теста.
email_time = group_data.loc[
    group_data["channel"] == "email",
    "resolution_hours"
].dropna()

# Аналогично формируем независимую выборку для chat.
chat_time = group_data.loc[
    group_data["channel"] == "chat",
    "resolution_hours"
].dropna()

# Заранее задаём уровень значимости.
alpha = 0.05

# equal_var=False включает Welch t-test,
# который не требует считать дисперсии двух групп одинаковыми.
test_result = stats.ttest_ind(
    email_time,
    chat_time,
    equal_var=False
)

# Выводим размер каждой выборки и результат теста.
print("Количество email:", len(email_time))
print("Количество chat:", len(chat_time))
print("t-statistic:", test_result.statistic)
print("p-value:", test_result.pvalue)

# Сравниваем p-value с заранее выбранным alpha.
if test_result.pvalue < alpha:
    print("Решение: есть основание отвергнуть H0 при alpha = 0.05.")
else:
    print("Решение: недостаточно оснований отвергнуть H0 при alpha = 0.05.")


## 10. Интерпретация результата теста

Теперь переведите статистический результат на понятный аналитический язык.

### Что обязательно включить в вывод

1. Что сравнивалось.
2. Какой уровень значимости использовался.
3. Какое p-value получено.
4. Какое решение принято по H0.
5. Какое ограничение есть у вывода.

Не используйте формулировки:

- «гипотеза доказана»;
- «вероятность истинности H0 равна p-value»;
- «канал является причиной различия».

Корректнее писать: «есть основание отвергнуть H0» или «недостаточно оснований отвергнуть H0».

**Шаблон:**

> На данных за выбранный период была проверена гипотеза о различии среднего времени решения обращений между каналами `email` и `chat`. При уровне значимости 0.05 p-value составило ... . Это даёт / не даёт основание отвергнуть H0. Вывод следует использовать осторожно, потому что ... .


## 11. Подготовка временного ряда

Теперь переходим от сравнения групп к анализу динамики.

### Что делает этот шаг

1. Сортирует обращения по дате.
2. Делает `created_at` временным индексом.
3. Объединяет обращения в недельные интервалы с помощью `resample("W")`.
4. Считает число уникальных обращений в каждой неделе.

### Почему важен порядок времени

Во временном ряду строки нельзя произвольно перемешивать. Прошлые периоды должны идти раньше будущих, иначе график и прогноз потеряют смысл.

### Что должно получиться

Таблица с временным индексом и одним показателем `tickets_count`. Каждая строка соответствует одной неделе.


In [ ]:
# Сортируем обращения по дате.
# Для временного ряда порядок наблюдений принципиален.
time_data = analysis_data.sort_values("created_at").copy()

# Строим недельный временной ряд:
# 1) делаем дату индексом;
# 2) объединяем строки в недельные интервалы;
# 3) считаем число уникальных обращений в каждой неделе.
weekly_tickets = (
    time_data
    .set_index("created_at")
    .resample("W")
    .agg(tickets_count=("ticket_id", "nunique"))
)

# Показываем первые недели и общий размер временного ряда.
display(weekly_tickets.head())
print("Количество недель:", len(weekly_tickets))


## 12. Визуализация динамики и скользящее среднее

### Что такое скользящее среднее

Для каждой недели берётся среднее значение по окну из четырёх недель. Такое сглаживание уменьшает влияние случайных колебаний и помогает увидеть общий тренд.

### Что показывает график

- линия `Факт` — реальное число обращений по неделям;
- линия `Скользящее среднее 4 недели` — сглаженная динамика.

### Как читать результат

Обратите внимание:

- растёт или снижается общий уровень;
- есть ли резкие пики;
- повторяются ли похожие колебания;
- насколько сглаженная линия отстаёт от фактических изменений.

Скользящее среднее используется для анализа динамики. Оно не является гарантией будущего поведения.


In [ ]:
# rolling(window=4) создаёт скользящее окно из четырёх недель,
# mean() рассчитывает среднее внутри каждого окна.
# Для первых трёх недель значение будет NaN, потому что полное окно ещё не накоплено.
weekly_tickets["moving_avg_4"] = (
    weekly_tickets["tickets_count"]
    .rolling(window=4)
    .mean()
)

# Создаём область графика.
plt.figure(figsize=(10, 5))

# Строим фактическое количество обращений.
plt.plot(
    weekly_tickets.index,
    weekly_tickets["tickets_count"],
    marker="o",
    label="Факт"
)

# Добавляем сглаженную линию.
plt.plot(
    weekly_tickets.index,
    weekly_tickets["moving_avg_4"],
    marker="o",
    label="Скользящее среднее 4 недели"
)

# Добавляем понятные подписи и элементы оформления.
plt.title("Недельная динамика обращений")
plt.xlabel("Неделя")
plt.ylabel("Количество обращений")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 13. Baseline-прогноз

Baseline — это простой прогноз, с которым можно сравнивать более сложные модели.

### Как делятся данные

- `train` — все недели, кроме последних четырёх;
- `test` — последние четыре недели.

Такое деление сохраняет временной порядок: модель использует только прошлое и проверяется на более позднем периоде.

### Как строится прогноз

Для каждой недели тестового периода используется одно значение — среднее число обращений за последние четыре недели обучающей части.

### Что должно получиться

Таблица `comparison` с двумя столбцами:

- `fact` — фактическое значение;
- `forecast` — baseline-прогноз.

Чем ближе эти значения, тем меньше ошибка прогноза.


In [ ]:
# Берём исходный недельный ряд без пропусков.
series = weekly_tickets["tickets_count"].dropna()

# Все недели, кроме последних четырёх, используем как прошлые данные.
train = series.iloc[:-4]

# Последние четыре недели используем как тестовый период.
test = series.iloc[-4:]

# Рассчитываем одно baseline-значение:
# среднее количество обращений за последние четыре недели train.
baseline_value = train.tail(4).mean()

# Создаём прогноз той же длины и с теми же датами, что и test.
baseline_forecast = pd.Series(
    baseline_value,
    index=test.index,
    name="forecast"
)

# Объединяем факт и прогноз в одну таблицу для сравнения.
comparison = pd.DataFrame({
    "fact": test,
    "forecast": baseline_forecast
})

# Ожидаются четыре строки — по одной на каждую тестовую неделю.
display(comparison)


## 14. Оценка ошибки прогноза

Прогноз нельзя оценивать только визуально. Для этого используются числовые метрики.

### MAE

Средняя абсолютная ошибка. Показывает, на сколько обращений прогноз в среднем отличается от факта.

Пример: `MAE = 6` означает, что прогноз в среднем ошибается примерно на 6 обращений за неделю.

### RMSE

Корень из средней квадратичной ошибки. Сильнее реагирует на крупные промахи, чем MAE.

### Сохранение результата

Таблица `comparison` сохраняется в папку `outputs` как `forecast_comparison.csv`. Этот файл можно использовать в отчёте или передать на проверку.

### Что проверить

- метрики являются неотрицательными числами;
- файл успешно сохранён;
- в `comparison` есть четыре строки тестового периода.


In [ ]:
# MAE — среднее абсолютное расстояние между фактом и прогнозом.
mae = mean_absolute_error(
    comparison["fact"],
    comparison["forecast"]
)

# MSE сначала возводит ошибки в квадрат.
mse = mean_squared_error(
    comparison["fact"],
    comparison["forecast"]
)

# RMSE возвращает ошибку к исходным единицам показателя.
rmse = np.sqrt(mse)

print("MAE:", mae)
print("RMSE:", rmse)

# Формируем путь к итоговому CSV.
output_path = OUTPUT_DIR / "forecast_comparison.csv"

# Сохраняем индекс с датами, чтобы тестовые недели не потерялись.
comparison.to_csv(output_path, index=True)
print("Файл сохранён:", output_path)


## 15. График факта и прогноза

Этот график позволяет визуально сравнить прогноз с фактическими значениями на тестовом периоде.

### Как читать график

- совпадающие или близкие линии означают небольшую ошибку;
- большое расстояние между линиями показывает промах прогноза;
- постоянная горизонтальная линия прогноза ожидаема: baseline использует одно среднее значение для всех четырёх недель.

Визуальная проверка дополняет MAE и RMSE, но не заменяет их.


In [ ]:
# Создаём отдельный график для тестового периода.
plt.figure(figsize=(10, 5))

# Фактические значения последних четырёх недель.
plt.plot(
    comparison.index,
    comparison["fact"],
    marker="o",
    label="Факт"
)

# Baseline-прогноз на те же недели.
plt.plot(
    comparison.index,
    comparison["forecast"],
    marker="o",
    label="Baseline-прогноз"
)

# Оформляем график.
plt.title("Сравнение факта и baseline-прогноза")
plt.xlabel("Неделя")
plt.ylabel("Количество обращений")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 16. ARIMA-прогноз и сравнение с baseline

Теперь построим второй прогноз на том же обучающем и тестовом периоде.

### Что такое ARIMA

ARIMA описывается тремя параметрами `order=(p, d, q)`:

- `p` — сколько прошлых значений ряда учитывается авторегрессионной частью;
- `d` — сколько раз ряд дифференцируется, чтобы анализировать изменения, а не только исходный уровень;
- `q` — сколько прошлых ошибок прогноза учитывает модель.

В учебном примере используем `ARIMA(1, 1, 1)`. Это **фиксированная демонстрационная конфигурация**, а не автоматически найденная лучшая модель.

### Логика расчёта

1. Модель обучается только на `train`.
2. Количество шагов прогноза равно длине `test`.
3. ARIMA-прогноз сравнивается с теми же фактическими значениями, что и baseline.
4. Для обеих моделей рассчитываются MAE и RMSE.

### Как интерпретировать сравнение

Меньшая ошибка на тестовом периоде означает, что модель лучше описала именно этот отложенный фрагмент данных. Это не гарантирует, что она всегда будет лучше на новых периодах: история короткая, параметры не подбирались, сезонность отдельно не моделируется.

In [ ]:
# Проверяем, достаточно ли наблюдений для учебного расчёта ARIMA.
# На очень коротком ряду параметры модели оцениваются нестабильно.
if len(train) < 8:
    raise ValueError(
        "Для учебного ARIMA-прогноза требуется хотя бы 8 недель в train. "
        f"Сейчас доступно: {len(train)}."
    )

# Создаём модель ARIMA(1, 1, 1).
# В модель передаём только train, поэтому значения test не участвуют в обучении.
arima_model = ARIMA(
    train.astype(float),
    order=(1, 1, 1)
)

# fit() оценивает параметры модели по обучающей части временного ряда.
arima_result = arima_model.fit()

# Строим прогноз ровно на столько недель, сколько находится в test.
arima_values = arima_result.forecast(steps=len(test))

# Явно назначаем прогнозу даты тестового периода.
# np.asarray() помогает избежать несовпадения индексов при объединении таблиц.
arima_forecast = pd.Series(
    np.asarray(arima_values, dtype=float),
    index=test.index,
    name="arima_forecast"
)

# Добавляем ARIMA-прогноз в общую таблицу факта и baseline.
comparison["arima_forecast"] = arima_forecast

# Считаем ошибки ARIMA на том же тестовом периоде.
arima_mae = mean_absolute_error(
    comparison["fact"],
    comparison["arima_forecast"]
)
arima_rmse = np.sqrt(
    mean_squared_error(
        comparison["fact"],
        comparison["arima_forecast"]
    )
)

# Собираем метрики двух моделей в одну таблицу.
model_metrics = pd.DataFrame({
    "model": ["Baseline: среднее последних 4 недель", "ARIMA(1, 1, 1)"],
    "MAE": [mae, arima_mae],
    "RMSE": [rmse, arima_rmse]
}).sort_values("MAE", ignore_index=True)

print("ARIMA AIC:", arima_result.aic)
display(comparison)
display(model_metrics)

# Строим единый график, чтобы визуально сравнить факт и оба прогноза.
plt.figure(figsize=(10, 5))
plt.plot(comparison.index, comparison["fact"], marker="o", label="Факт")
plt.plot(comparison.index, comparison["forecast"], marker="o", label="Baseline")
plt.plot(comparison.index, comparison["arima_forecast"], marker="o", label="ARIMA(1, 1, 1)")
plt.title("Сравнение факта, baseline и ARIMA-прогноза")
plt.xlabel("Неделя")
plt.ylabel("Количество обращений")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Перезаписываем итоговую таблицу: теперь в ней есть оба прогноза.
comparison.to_csv(OUTPUT_DIR / "forecast_comparison.csv", index=True)
model_metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

print("Файл сохранён:", OUTPUT_DIR / "forecast_comparison.csv")
print("Файл сохранён:", OUTPUT_DIR / "model_metrics.csv")

## 17. Итоговый аналитический вывод

Итоговый вывод должен связывать две части работы: проверку гипотезы и прогнозирование временного ряда.

Заполните вывод по структуре:

1. Что анализировалось.
2. Какие данные и период использовались.
3. Какая гипотеза проверялась.
4. Почему был выбран Welch t-test.
5. Какое p-value получено.
6. Какое решение принято по H0.
7. Что видно по недельной динамике.
8. Как построен baseline-прогноз.
9. Как построен ARIMA(1, 1, 1)-прогноз.
10. Какие MAE и RMSE получены у каждой модели.
11. Какая модель показала меньшую ошибку на test.
12. Какие ограничения есть у анализа.

### Примеры ограничений

- сравниваются только два канала;
- не учитывается сложность обращения;
- возможны сезонные эффекты;
- история наблюдений ограничена;
- baseline не учитывает внешние события и изменение процессов;
- параметры ARIMA заданы как учебный пример и не подбирались;
- четыре тестовые недели дают ограниченную оценку качества.

**Место для вывода:**

...


## 18. Чек-лист завершения

Перед сдачей выполните `Kernel → Restart Kernel and Run All Cells` или аналогичную команду запуска всех ячеек. Это проверит, что notebook не зависит от скрытого состояния.

- [ ] Notebook запускается сверху вниз без ошибок.
- [ ] Используются относительные пути.
- [ ] Файл `support_tickets.csv` найден.
- [ ] Дата преобразована в `datetime`.
- [ ] Проверены пропуски, дубликаты и некорректные значения.
- [ ] H0 и H1 записаны явно.
- [ ] В таблице сравнения есть `email` и `chat`.
- [ ] p-value сравнивается с `alpha = 0.05`.
- [ ] Вывод не содержит фразы «гипотеза доказана».
- [ ] Временной ряд построен по неделям.
- [ ] Train/test split выполнен по времени.
- [ ] Forecast сравнен с fact.
- [ ] MAE и RMSE baseline посчитаны и объяснены словами.
- [ ] ARIMA обучена только на `train`.
- [ ] ARIMA-прогноз построен на даты `test`.
- [ ] Ошибки baseline и ARIMA сравниваются на одном тестовом периоде.
- [ ] `forecast_comparison.csv` и `model_metrics.csv` сохранены в `outputs`.
- [ ] Итоговый вывод содержит ограничения.
